# Build and analyze an SAE feature dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/sae_features.ipynb)

Download the held-out validation split from `jxliu2/idiom-data`, build a reusable feature
dataset for its IDRs, rank features, and inspect their strongest sequences and residue-level
activation traces. The SAE loads its recorded host model and layer automatically.

A GPU is recommended. In Colab, select **Runtime → Change runtime type → GPU** before running.
`DEVICE = "auto"` falls back to CPU.

In [ ]:
# Install into this notebook's Python environment if IDiom is unavailable.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/rotskoff-group/idiom.git",
    ])

from huggingface_hub import hf_hub_download
import idiom

print("IDiom loaded from", idiom.__file__)

In [ ]:
from pathlib import Path

SAE = "jxliu2/idiomsae-300M-L18-k32"  # Hub ID or local release directory
DEVICE = "auto"
INPUT_FASTA = None  # None downloads the held-out validation split; or set a local record FASTA.
MAX_RECORDS = None  # Use all valid records; set an integer for a smaller run.
OUT_DIR = Path("sae_feature_dataset")  # Build overwrites dataset files in this directory.
BATCH_SIZE = 4  # Lower this if GPU memory is limited.
N_TOP = 10
FEATURE_ID = None  # None selects the feature with the largest activation.

## 1. Prepare a record FASTA

Each header ends in `_IDR_x-y`, with **1-based inclusive** coordinates in the full protein:

```text
>protein_A_IDR_3-9
MEDSKVDNRPQACDEFG
```

For isolated IDRs, use `_IDR_1-<sequence_length>`. The reader converts spans to 0-based,
half-open coordinates and skips malformed records and non-canonical sequences.
By default, this notebook downloads `training_sequences/validation.fasta` from
`jxliu2/idiom-data` and analyzes all valid records. Set `INPUT_FASTA` to use your own data.
The validation split contains approximately 271,000 records; set `MAX_RECORDS` to a smaller
integer (for example, 1,000) for a quick run or a memory-limited environment.

In [ ]:
from itertools import islice

from idiom.data.io import read_records

fasta = Path(INPUT_FASTA) if INPUT_FASTA is not None else Path(hf_hub_download(
    "jxliu2/idiom-data", "training_sequences/validation.fasta", repo_type="dataset",
))
records = list(islice(read_records(fasta), MAX_RECORDS))
if not records:
    raise ValueError("No valid records found; check the FASTA headers and sequences.")

print(f"Loaded {len(records)} records from {fasta}")
for record in records[:3]:
    idr = record.full_seq[record.idr_start:record.idr_end]
    print(f"{record.accession}: {len(record.full_seq)} residues, IDR length {len(idr)}")

## 2. Build the feature dataset

The released SAE uses **unprompted IDRs** (`region="idr"`, `fim_mode="unprompted"`), so protein
flanks are excluded. A different SAE supplies its own layer, region, and prompt format.

The builder saves top-k feature IDs and values for each selected residue, sequence indices,
FIM-string positions, the strings themselves, and dataset metadata. Building holds the selected
records and output arrays in memory. Full validation analysis can require substantial host RAM
and runtime; reduce `MAX_RECORDS` to limit total memory and `BATCH_SIZE` to limit GPU memory.

In [ ]:
from idiom import IDiomSAE

sae = IDiomSAE.from_pretrained(SAE, device=DEVICE)
print(f"Host: {sae.host_model}; layer: {sae.layer}")
print(f"Region: {sae.region}; FIM mode: {sae.fim_mode}; latents: {sae.sae.num_latents}")

lengths = [
    r.idr_end - r.idr_start if sae.fim_mode == "unprompted" else len(r.full_seq)
    for r in records
]
if max(lengths) + 4 > sae.model.cfg.max_seq_len:
    raise ValueError("An input exceeds the host model's context length, including START and FIM markers.")

sae.build_feature_dataset(records, OUT_DIR, batch_size=BATCH_SIZE)
print(f"Saved feature dataset to {OUT_DIR.resolve()}")

## 3. Browse features

Open the saved dataset without rerunning the models. Arrays are memory-mapped by default.
`feature_ranking()` returns three arrays indexed by **feature ID**: maximum activation,
total activation, and the number of residue rows with positive activation.
These are descriptive rankings, not statistical enrichment tests.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from idiom.sae.features import FeatureDataset

# This section can also inspect a previously built OUT_DIR.
dataset = FeatureDataset(OUT_DIR)
maximum, total, count = dataset.feature_ranking()
active = np.flatnonzero(count > 0)
ranked = active[np.argsort(-maximum[active], kind="stable")]
print(f"{dataset.n_seqs} sequences, {dataset.top_indices.shape[0]} residue rows")
print(f"{len(active)} of {dataset.num_latents} features have positive activations")
print(f"{'feature':>8} {'maximum':>10} {'total':>12} {'firing rows':>12}")
for feature in ranked[:N_TOP]:
    print(f"{feature:8d} {maximum[feature]:10.3f} {total[feature]:12.3f} {count[feature]:12d}")

In [ ]:
top = ranked[:N_TOP]
if len(top):
    fig, ax = plt.subplots(figsize=(8, 3), constrained_layout=True)
    ax.bar(np.arange(len(top)), maximum[top])
    ax.set_xticks(np.arange(len(top)), labels=top)
    ax.set(xlabel="Feature ID", ylabel="Maximum activation", title="Strongest features")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()
else:
    print("No features fired on this dataset.")

For an interactive browser in a **local repository clone**, run this optional command in a
terminal (replace the dataset path with the absolute `OUT_DIR` printed above):

```bash
streamlit run src/idiom/sae/features/feature_viewer.py -- --features /path/to/sae_feature_dataset
```

The viewer ranks by maximum activation, total activation, or firing count and shades residues
in the strongest sequences. The inline analysis below works in both Colab and local notebooks.

## 4. Inspect a feature's sequences and activation traces

Set `FEATURE_ID` in the parameter cell to examine a specific feature; otherwise use the top-ranked
one. Rank sequences by `"peak"` activation or `"fraction"` of selected residues with positive
activation. Sequence IDs index `dataset.strings`; they are not FASTA accessions.

In [ ]:
if FEATURE_ID is None and not len(ranked):
    raise ValueError("No active features to inspect; try another input dataset.")
feature_id = int(ranked[0]) if FEATURE_ID is None else int(FEATURE_ID)
if not 0 <= feature_id < dataset.num_latents:
    raise ValueError(f"FEATURE_ID must be between 0 and {dataset.num_latents - 1}.")

sequence_ids, scores = dataset.top_sequences(feature_id, n=N_TOP, sort_by="peak")
global_max, peaks, fractions = dataset.feature_stats(feature_id)
print(f"Feature {feature_id}: maximum={global_max:.3f}")
for seq_id, score in zip(sequence_ids, scores):
    print(f"Sequence {seq_id}: peak={score:.3f}, firing fraction={fractions[seq_id]:.3f}")
    print(dataset.sequence(int(seq_id)))
if not len(sequence_ids):
    print("This feature has no positive activations; choose another FEATURE_ID.")

`trace()` returns positions in the stored **FIM string**, including the `1`, `3`, and `2` markers
but excluding START. These are not original protein coordinates. For the released unprompted
SAE, the string is `132{IDR}` and the first residue is at position 3.

Plots show only stored residue rows. Markers and unselected regions have no activation rows;
an unselected feature at a stored residue has activation zero.

In [ ]:
for seq_id in sequence_ids[:3]:
    seq_id = int(seq_id)
    positions, activations = dataset.trace(seq_id, feature_id)
    fim_string = dataset.sequence(seq_id)

    fig, ax = plt.subplots(figsize=(10, 2.5), constrained_layout=True)
    ax.plot(positions, activations, linewidth=1)
    if len(positions) <= 60:
        ax.set_xticks(positions, labels=[fim_string[int(p)] for p in positions])
        ax.set_xlabel("Residue in FIM order")
    else:
        ax.set_xlabel("Position in stored FIM string (0-based)")
    ax.set_ylabel("Activation")
    ax.set_title(f"Feature {feature_id} · sequence {seq_id}")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()

    if not len(activations):
        continue
    peak_row = int(activations.argmax())
    peak_pos = int(positions[peak_row])
    print(f"Sequence {seq_id}: peak at FIM position {peak_pos} ({fim_string[peak_pos]})")

## Next: compare against a background

Use [feature_enrichment.ipynb](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/feature_enrichment.ipynb)
to find features enriched in a positive sequence set against a background, filter boundary
artifacts, and export a feature signature.